# Inference Speed Benchmark — 154-Class Models

Measures **single-sample CPU inference latency** for the 154-class experiment models.

## What is measured
| Metric | Description |
|---|---|
| Mean (ms) | Average latency over N trials |
| Median (ms) | Robust central latency (less affected by outliers) |
| Std (ms) | Trial-to-trial variability |
| Min / Max (ms) | Best and worst single-sample latency |
| P95 (ms) | 95th-percentile worst-case latency |
| Throughput (FPS) | Samples per second at mean latency |
| Model size (MB) | Disk size of the saved .pkl file |

## Why CPU-only
All benchmarks run with `CUDA_VISIBLE_DEVICES=-1` (GPU disabled) to simulate  
deployment on an edge device (Raspberry Pi / laptop without GPU).

## 154-Class Models
| Model | Input | Notes |
|---|---|---|
| InceptionTime | ASL_154_test_cif.npz | 126 ch · 40 frames · hands only · 5-ensemble |
| Random Forest | ASL_154_test_rf.csv | 100 trees · 630 features · hands only |
| Logistic Regression | ASL_154_test_rf.csv | lbfgs · C=1.0 · 630 features · hands only |
| CIF (50 trees) | ASL_154_test_cif.npz | 50 trees · 126 ch · hands only |

Results saved to `results/analysis/`.

In [1]:
# ── Cell 0: Mount Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ── Cell 1b: Install sktime (required to load CIF .pkl files) ────────────
import subprocess, sys

print('Installing sktime...')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'sktime==0.40.1', 'scikit-base==0.13.2',
    '-q', '--disable-pip-version-check'
])
print('✅ sktime ready')

# Verify it imports correctly
import sktime
print(f'   sktime version: {sktime.__version__}')

Installing sktime...
✅ sktime ready
   sktime version: 0.40.1


In [3]:
# ── Cell 1: Imports & paths ──────────────────────────────────────────────
import os, sys, time, gc
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

# Force CPU — simulates edge deployment (Raspberry Pi / no-GPU laptop)
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

PROJECT_DIR  = '/content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2'
RESULTS_DIR  = os.path.join(PROJECT_DIR, 'results')
ANALYSIS_DIR = os.path.join(RESULTS_DIR, 'analysis')
os.makedirs(ANALYSIS_DIR, exist_ok=True)

# Add project to path (needed to import inceptiontime_classifier.py)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f'Project  : {PROJECT_DIR}')
print(f'Results  : {RESULTS_DIR}')
print(f'Analysis : {ANALYSIS_DIR}')
print(f'Drive OK : {os.path.exists(RESULTS_DIR)}')

Project  : /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2
Results  : /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results
Analysis : /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/analysis
Drive OK : True


In [4]:
# ── Cell 2: Benchmark registry ───────────────────────────────────────────
# 154-class experiment only.
#
#  Model               | Save format                          | Input
#  ────────────────────┼──────────────────────────────────────┼────────────────────────
#  InceptionTime       | 5x .keras files (Keras ensemble)     | ASL_154_test_cif.npz
#  Random Forest       | rf_154_model.pkl  (joblib)           | ASL_154_test_rf.csv
#  Logistic Regression | lr_154_model.pkl  (joblib)           | ASL_154_test_rf.csv
#  CIF (50 trees)      | cif_model.pkl (joblib)               | ASL_154_test_cif.npz
#
# InceptionTime was saved as a 5-model Keras ensemble (best_model_seed{seed}.keras).
# There is no single .pkl for IT. The benchmark loads all 5 .keras files and times
# the full ensemble predict (mean of 5 softmax outputs) per sample.

P = PROJECT_DIR  # shorthand
R = RESULTS_DIR

SEEDS = [42, 123, 456, 789, 1011]   # same seeds used during training

BENCHMARK_REGISTRY = [

    # ── InceptionTime: Keras ensemble ────────────────────────────────────
    dict(experiment='154-class',   n_classes=154,
         model='InceptionTime',      model_type='keras_ensemble',
         result_dir='inc_154',
         keras_paths=[f'{R}/inc_154/best_model_seed{s}.keras' for s in SEEDS],
         npz_path=f'{P}/data/npz/ASL_154_test_cif.npz',
         special='126 ch · 40 frames · hands only · 5-model Keras ensemble'),

    # ── Sklearn/joblib models ─────────────────────────────────────────────
    dict(experiment='154-class',   n_classes=154,
         model='Random Forest',      model_type='tabular',
         result_dir='rf_154',        pkl_name='rf_154_model.pkl',
         agg_csv=f'{P}/data/csv/ASL_154_test_rf.csv',
         special='100 trees · 630 features · hands only'),

    dict(experiment='154-class',   n_classes=154,
         model='Logistic Regression', model_type='tabular',
         result_dir='lr_154',        pkl_name='lr_154_model.pkl',
         agg_csv=f'{P}/data/csv/ASL_154_test_rf.csv',
         special='lbfgs · C=1.0 · 630 features · hands only'),

    dict(experiment='154-class',   n_classes=154,
         model='CIF (50 trees)',     model_type='timeseries',
         result_dir='cif_154',       pkl_name='cif_model.pkl',
         npz_path=f'{P}/data/npz/ASL_154_test_cif.npz',
         special='50 trees · 126 ch · hands only'),
]

print(f'Registry: {len(BENCHMARK_REGISTRY)} models (154-class experiment).')
print()
print('Verifying files exist...')
for slot in BENCHMARK_REGISTRY:
    src = slot.get('npz_path', slot.get('agg_csv', 'N/A'))
    src_ok = '\u2705' if os.path.exists(src) else '\u274c MISSING input'
    if slot['model_type'] == 'keras_ensemble':
        n_found  = sum(os.path.exists(p) for p in slot['keras_paths'])
        mdl_ok   = f'\u2705 {n_found}/{len(slot["keras_paths"])} .keras' if n_found == len(slot["keras_paths"]) else f'\u274c {n_found}/{len(slot["keras_paths"])} .keras MISSING'
    else:
        pkl = os.path.join(RESULTS_DIR, slot['result_dir'], slot['pkl_name'])
        mdl_ok = '\u2705' if os.path.exists(pkl) else '\u274c MISSING pkl'
    print(f'  {slot["experiment"]:12s} / {slot["model"]:25s}  model:{mdl_ok}  input:{src_ok}')


Registry: 4 models (154-class experiment).

Verifying files exist...
  154-class    / InceptionTime              model:✅ 5/5 .keras  input:✅
  154-class    / Random Forest              model:✅  input:✅
  154-class    / Logistic Regression        model:✅  input:✅
  154-class    / CIF (50 trees)             model:✅  input:✅


In [5]:
# ── Cell 3: Helpers ──────────────────────────────────────────────────────
import os, time, gc
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf

# Ensure CPU-only inference (simulates edge deployment)
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
tf.config.set_visible_devices([], 'GPU')

_tab_cache = {}   # agg_csv path  → 1-row DataFrame
_ts_cache  = {}   # npz_path      → numpy array (1, C, T) — raw NPZ format


def get_tabular_sample(agg_csv):
    if agg_csv not in _tab_cache:
        if not os.path.exists(agg_csv):
            return None
        df = pd.read_csv(agg_csv)
        drop = [c for c in ['video_name','label','gloss','filename'] if c in df.columns]
        _tab_cache[agg_csv] = df.drop(columns=drop).iloc[[0]]
    return _tab_cache[agg_csv]


def get_ts_sample(npz_path, transpose_for_keras=False):
    """Load one sample from the NPZ.
    NPZ stores X as (n, 126, 40) — shape (n_channels, n_frames).
    CIF/sktime expects (1, n_channels, n_frames)  → no transpose.
    Keras IT expects  (1, n_frames,   n_channels) → transpose=True.
    """
    cache_key = (npz_path, transpose_for_keras)
    if cache_key not in _ts_cache:
        if not os.path.exists(npz_path):
            return None
        d   = np.load(npz_path, allow_pickle=True)
        arr = d['X'][[0]]              # (1, 126, 40)
        if transpose_for_keras:
            arr = arr.transpose(0, 2, 1).astype(np.float32)  # (1, 40, 126)
        print(f'    [cache] {os.path.basename(npz_path)}  shape={arr.shape}'
              + (' [transposed for Keras]' if transpose_for_keras else ''))
        _ts_cache[cache_key] = arr
    return _ts_cache[cache_key]


def load_keras_ensemble(keras_paths):
    """Load all .keras files and return as a list of Keras models."""
    models = []
    for p in keras_paths:
        if not os.path.exists(p):
            raise FileNotFoundError(f'Keras model not found: {p}')
        models.append(tf.keras.models.load_model(p))
        print(f'    Loaded {os.path.basename(p)}')
    return models


def load_sklearn_model(pkl_path, is_cif=False):
    """Load a joblib .pkl (RF, LR, or CIF)."""
    return joblib.load(pkl_path)


def run_benchmark_keras(keras_models, X_sample, n_trials=100):
    """Benchmark the full 5-model Keras ensemble (mean of softmax outputs).
    X_sample shape: (1, 40, 126) — already transposed for Keras.
    5 warm-up calls (each calling all 5 models) excluded from stats.
    """
    def predict_ensemble(X):
        preds = [m.predict(X, verbose=0) for m in keras_models]
        return np.mean(preds, axis=0)

    for _ in range(5):            # warm-up
        predict_ensemble(X_sample)

    times = []
    for _ in range(n_trials):
        t0 = time.perf_counter()
        predict_ensemble(X_sample)
        times.append((time.perf_counter() - t0) * 1000)

    arr = np.array(times)
    return dict(
        mean_ms   = float(np.mean(arr)),
        median_ms = float(np.median(arr)),
        std_ms    = float(np.std(arr)),
        min_ms    = float(np.min(arr)),
        max_ms    = float(np.max(arr)),
        p95_ms    = float(np.percentile(arr, 95)),
    )


def run_benchmark_sklearn(model, X_sample, n_trials=100):
    """Benchmark a single sklearn predict() call.
    5 warm-up calls excluded from stats.
    """
    for _ in range(5):            # warm-up
        model.predict(X_sample)

    times = []
    for _ in range(n_trials):
        t0 = time.perf_counter()
        model.predict(X_sample)
        times.append((time.perf_counter() - t0) * 1000)

    arr = np.array(times)
    return dict(
        mean_ms   = float(np.mean(arr)),
        median_ms = float(np.median(arr)),
        std_ms    = float(np.std(arr)),
        min_ms    = float(np.min(arr)),
        max_ms    = float(np.max(arr)),
        p95_ms    = float(np.percentile(arr, 95)),
    )


print('Helpers ready.')


Helpers ready.


In [ ]:
# ── Cell 4: Run benchmarks ───────────────────────────────────────────────
N_TRIALS = 100

print(f'CPU-only inference benchmark  ({N_TRIALS} trials + 5 warm-up per model)')
print(f'GPU disabled: CUDA_VISIBLE_DEVICES={os.environ["CUDA_VISIBLE_DEVICES"]}')
print('=' * 72)

rows = []

for slot in BENCHMARK_REGISTRY:
    label = f"{slot['experiment']} / {slot['model']}"

    # ── InceptionTime: Keras ensemble ────────────────────────────────────
    if slot['model_type'] == 'keras_ensemble':
        missing_keras = [p for p in slot['keras_paths'] if not os.path.exists(p)]
        if missing_keras:
            print(f'  [SKIP — {len(missing_keras)} .keras file(s) missing]  {label}')
            for p in missing_keras:
                print(f'    ✗ {p}')
            rows.append({'Experiment': slot['experiment'], 'Model': slot['model'],
                         'Classes': slot['n_classes'], 'Input type': 'keras_ensemble',
                         'Special conditions': slot['special'],
                         'Model size (MB)': None,
                         'Mean (ms)': None, 'Median (ms)': None, 'Std (ms)': None,
                         'Min (ms)': None,  'Max (ms)': None,    'P95 (ms)': None,
                         'Throughput (FPS)': None})
            continue

        # Combined size of all 5 .keras files
        total_mb = sum(os.path.getsize(p) for p in slot['keras_paths']) / (1024 ** 2)

        # Input: NPZ transposed to (1, 40, 126) for Keras
        X_sample = get_ts_sample(slot['npz_path'], transpose_for_keras=True)
        if X_sample is None:
            print(f'  [SKIP — NPZ missing]  {label}')
            continue

        print(f'\n  Benchmarking: {label}')
        print(f'    Ensemble size : {total_mb:.1f} MB  ({len(slot["keras_paths"])} models × ~{total_mb/len(slot["keras_paths"]):.1f} MB)')
        print(f'    Input shape   : {X_sample.shape}  [keras_ensemble, transposed]')
        print(f'    Special       : {slot["special"]}')
        print(f'    Loading {len(slot["keras_paths"])} Keras models...')

        keras_models = load_keras_ensemble(slot['keras_paths'])
        stats = run_benchmark_keras(keras_models, X_sample, n_trials=N_TRIALS)
        fps   = 1000.0 / stats['mean_ms'] if stats['mean_ms'] > 0 else None

        print(f'    Mean={stats["mean_ms"]:.2f} ms  '
              f'Median={stats["median_ms"]:.2f} ms  '
              f'Std={stats["std_ms"]:.2f} ms  '
              f'P95={stats["p95_ms"]:.2f} ms  '
              f'FPS={fps:.1f}')

        rows.append({
            'Experiment':         slot['experiment'],
            'Model':              slot['model'],
            'Classes':            slot['n_classes'],
            'Input type':         'keras_ensemble',
            'Special conditions': slot['special'],
            'Model size (MB)':    round(total_mb, 1),
            'Mean (ms)':          round(stats['mean_ms'],   2),
            'Median (ms)':        round(stats['median_ms'], 2),
            'Std (ms)':           round(stats['std_ms'],    2),
            'Min (ms)':           round(stats['min_ms'],    2),
            'Max (ms)':           round(stats['max_ms'],    2),
            'P95 (ms)':           round(stats['p95_ms'],    2),
            'Throughput (FPS)':   round(fps, 1) if fps else None,
        })

        del keras_models; gc.collect()
        continue

    # ── sklearn / joblib models (RF, LR, CIF) ────────────────────────────
    rdir     = os.path.join(RESULTS_DIR, slot['result_dir'])
    pkl_path = os.path.join(rdir, slot['pkl_name'])

    if not os.path.exists(pkl_path):
        print(f'  [SKIP — pkl missing]  {label}')
        rows.append({'Experiment': slot['experiment'], 'Model': slot['model'],
                     'Classes': slot['n_classes'], 'Input type': slot['model_type'],
                     'Special conditions': slot['special'],
                     'Model size (MB)': None,
                     'Mean (ms)': None, 'Median (ms)': None, 'Std (ms)': None,
                     'Min (ms)': None,  'Max (ms)': None,    'P95 (ms)': None,
                     'Throughput (FPS)': None})
        continue

    model_mb = os.path.getsize(pkl_path) / (1024 ** 2)

    if slot['model_type'] == 'tabular':
        X_sample = get_tabular_sample(slot.get('agg_csv', ''))
    else:
        X_sample = get_ts_sample(slot.get('npz_path', ''), transpose_for_keras=False)

    if X_sample is None:
        print(f'  [SKIP — input missing]  {label}')
        rows.append({'Experiment': slot['experiment'], 'Model': slot['model'],
                     'Classes': slot['n_classes'], 'Input type': slot['model_type'],
                     'Special conditions': slot['special'],
                     'Model size (MB)': round(model_mb, 1),
                     'Mean (ms)': None, 'Median (ms)': None, 'Std (ms)': None,
                     'Min (ms)': None,  'Max (ms)': None,    'P95 (ms)': None,
                     'Throughput (FPS)': None})
        continue

    print(f'\n  Benchmarking: {label}')
    print(f'    Model size  : {model_mb:.1f} MB')
    print(f'    Input shape : {X_sample.shape}  [{slot["model_type"]}]')
    print(f'    Special     : {slot["special"]}')

    model  = load_sklearn_model(pkl_path)
    stats  = run_benchmark_sklearn(model, X_sample, n_trials=N_TRIALS)
    fps    = 1000.0 / stats['mean_ms'] if stats['mean_ms'] > 0 else None

    print(f'    Mean={stats["mean_ms"]:.2f} ms  '
          f'Median={stats["median_ms"]:.2f} ms  '
          f'Std={stats["std_ms"]:.2f} ms  '
          f'P95={stats["p95_ms"]:.2f} ms  '
          f'FPS={fps:.1f}')

    rows.append({
        'Experiment':         slot['experiment'],
        'Model':              slot['model'],
        'Classes':            slot['n_classes'],
        'Input type':         slot['model_type'],
        'Special conditions': slot['special'],
        'Model size (MB)':    round(model_mb, 1),
        'Mean (ms)':          round(stats['mean_ms'],   2),
        'Median (ms)':        round(stats['median_ms'], 2),
        'Std (ms)':           round(stats['std_ms'],    2),
        'Min (ms)':           round(stats['min_ms'],    2),
        'Max (ms)':           round(stats['max_ms'],    2),
        'P95 (ms)':           round(stats['p95_ms'],    2),
        'Throughput (FPS)':   round(fps, 1) if fps else None,
    })

    del model; gc.collect()

speed_df = pd.DataFrame(rows)
print(f'\n\n\u2705 Done. {speed_df["Mean (ms)"].notna().sum()} / {len(speed_df)} models benchmarked.')


CPU-only inference benchmark  (100 trials + 5 warm-up per model)
GPU disabled: CUDA_VISIBLE_DEVICES=-1
    [cache] ASL_154_test_cif.npz  shape=(1, 40, 126) [transposed for Keras]

  Benchmarking: 154-class / InceptionTime
    Ensemble size : 29.2 MB  (5 models × ~5.8 MB)
    Input shape   : (1, 40, 126)  [keras_ensemble, transposed]
    Special       : 126 ch · 40 frames · hands only · 5-model Keras ensemble
    Loading 5 Keras models...
    Loaded best_model_seed42.keras
    Loaded best_model_seed123.keras
    Loaded best_model_seed456.keras
    Loaded best_model_seed789.keras
    Loaded best_model_seed1011.keras


    Mean=447.01 ms  Median=441.84 ms  Std=34.88 ms  P95=472.02 ms  FPS=2.2

  Benchmarking: 154-class / Random Forest
    Model size  : 105.4 MB
    Input shape : (1, 630)  [tabular]
    Special     : 100 trees · 630 features · hands only


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_job

    Mean=31.55 ms  Median=31.22 ms  Std=2.37 ms  P95=35.16 ms  FPS=31.7

  Benchmarking: 154-class / Logistic Regression
    Model size  : 0.7 MB
    Input shape : (1, 630)  [tabular]
    Special     : lbfgs · C=1.0 · 630 features · hands only
    Mean=3.94 ms  Median=3.78 ms  Std=0.48 ms  P95=4.89 ms  FPS=253.7
    [cache] ASL_154_test_cif.npz  shape=(1, 126, 40)

  Benchmarking: 154-class / CIF (50 trees)
    Model size  : 85.4 MB
    Input shape : (1, 126, 40)  [timeseries]
    Special     : 50 trees · 126 ch · hands only


In [ ]:
# ── Cell 5: Full table — slowest → fastest ───────────────────────────────
from IPython.display import display

display_df = speed_df.copy()
display_df['_sort'] = display_df['Mean (ms)'].fillna(-1)
display_df = (display_df
              .sort_values('_sort', ascending=False)
              .drop(columns='_sort')
              .reset_index(drop=True))

# Speed-tier background colours (dark enough for #111 text)
def speed_bg(mean_ms):
    try: v = float(mean_ms)
    except (TypeError, ValueError): return 'background-color: #e0e0e0; color: #888888;'
    if v >= 500:  return 'background-color: #f5bcbc; color: #111111;'  # red   ≥ 500 ms
    if v >= 50:   return 'background-color: #f7d199; color: #111111;'  # amber 50–500 ms
    if v >= 10:   return 'background-color: #fff3b0; color: #111111;'  # yellow 10–50 ms
    return              'background-color: #b5ddb5; color: #111111;'   # green  < 10 ms

def row_colour(row):
    css = speed_bg(row.get('Mean (ms)', None))
    return [css] * len(row)

def bold_min(s):
    """Bold dark-green the lowest (fastest) numeric value in a column."""
    vals = []
    for v in s:
        try: vals.append(float(v))
        except (TypeError, ValueError): vals.append(None)
    best = min((v for v in vals if v is not None), default=None)
    return ['font-weight:bold; color:#0a4a0a;' if v == best else 'color:#111111;'
            for v in vals]

def bold_max(s):
    """Bold dark-green the highest (best FPS) value."""
    vals = []
    for v in s:
        try: vals.append(float(v))
        except (TypeError, ValueError): vals.append(None)
    best = max((v for v in vals if v is not None), default=None)
    return ['font-weight:bold; color:#0a4a0a;' if v == best else 'color:#111111;'
            for v in vals]

TH = ('background-color:#1a252f; color:#ffffff;'
      'padding:9px 12px; font-size:12px; text-align:left;'
      'border:1px solid #0d1a22; white-space:nowrap;')
TD = 'padding:8px 12px; font-size:13px; border-bottom:1px solid #888888;'

styled = (
    display_df
    .style
    .set_caption('CPU Inference Speed — 154-class models, sorted slowest → fastest')
    .apply(row_colour, axis=1)
    .apply(bold_min, subset=['Mean (ms)', 'Median (ms)', 'P95 (ms)'])
    .apply(bold_max, subset=['Throughput (FPS)'])
    .set_table_styles([
        {'selector':'th',      'props': TH},
        {'selector':'td',      'props': TD},
        {'selector':'caption', 'props': 'font-size:14px; font-weight:bold; color:#111111; padding:10px 0; text-align:left;'},
        {'selector':'table',   'props': 'border-collapse:collapse; width:100%;'},
    ])
    .hide(axis='index')
)

display(styled)
print()
print('Speed tier legend:')
print('  GREEN  — fast      < 10 ms')
print('  YELLOW — moderate  10–50 ms')
print('  AMBER  — slow      50–500 ms')
print('  RED    — very slow >= 500 ms')
print('  Bold dark-green = best value in that column')

In [ ]:
# ── Cell 6: Architecture summary — 154-class models ──────────────────────
# Summarises latency per model architecture for the 154-class experiment.

print('=' * 72)
print('  ARCHITECTURE SUMMARY — mean latency averaged across all experiments')
print('=' * 72)

arch = (
    speed_df[speed_df['Mean (ms)'].notna()]
    .groupby('Model')
    .agg(
        N            = ('Experiment', 'count'),
        Avg_mean_ms  = ('Mean (ms)',        'mean'),
        Avg_median_ms= ('Median (ms)',      'mean'),
        Avg_std_ms   = ('Std (ms)',         'mean'),
        Avg_p95_ms   = ('P95 (ms)',         'mean'),
        Avg_fps      = ('Throughput (FPS)', 'mean'),
        Avg_size_mb  = ('Model size (MB)',  'mean'),
    )
    .sort_values('Avg_mean_ms', ascending=False)
    .reset_index()
)

for c in arch.columns[2:]:
    arch[c] = arch[c].round(2)

arch.columns = ['Model', 'Exps', 'Avg Mean (ms)', 'Avg Median (ms)',
                'Avg Std (ms)', 'Avg P95 (ms)', 'Avg FPS', 'Avg Size (MB)']

display(
    arch.style
    .set_caption('Architecture speed summary — 154-class (slowest → fastest)')
    .apply(bold_min, subset=['Avg Mean (ms)'])
    .apply(bold_max, subset=['Avg FPS'])
    .set_properties(**{'font-size':'13px', 'color':'#111111', 'text-align':'left'})
    .set_table_styles([
        {'selector':'th', 'props': TH},
        {'selector':'td', 'props': TD},
        {'selector':'table', 'props': 'border-collapse:collapse; width:100%;'},
    ])
    .hide(axis='index')
)

print()
for _, r in arch.iterrows():
    print(f"  {r['Model']:<25} avg {r['Avg Mean (ms)']:>8.2f} ms  "
          f"{r['Avg FPS']:>7.1f} FPS  {r['Avg Size (MB)']:>8.1f} MB")

In [ ]:
# ── Cell 7: Latency bar chart — 154-class models ────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plot_df = speed_df[speed_df['Mean (ms)'].notna()].copy()
plot_df = plot_df.sort_values('Mean (ms)', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#E53935', '#FB8C00', '#FDD835', '#43A047']
bars = ax.barh(plot_df['Model'], plot_df['Mean (ms)'],
               color=colors[:len(plot_df)], edgecolor='black', alpha=0.85)
ax.set_xlabel('Mean Latency (ms)')
ax.set_title('CPU Inference Latency — 154-Class Models (slowest → fastest)')
for bar, row in zip(bars, plot_df.itertuples()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{row._6:.2f} ms  ({row._13:.1f} FPS)',
            va='center', fontsize=10)
ax.set_xlim(right=plot_df['Mean (ms)'].max() * 1.35)
plt.tight_layout()
CHART_PATH = os.path.join(ANALYSIS_DIR, 'inference_speed_154class_chart.png')
plt.savefig(CHART_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n\u2705 Chart saved → {CHART_PATH}')
plt.close()


In [ ]:
# ── Cell 8: Save ─────────────────────────────────────────────────────────

OUT_FULL  = os.path.join(ANALYSIS_DIR, 'inference_speed_154class.csv')
OUT_ARCH  = os.path.join(ANALYSIS_DIR, 'inference_speed_154class_by_architecture.csv')

speed_df.sort_values('Mean (ms)', ascending=False, na_position='last').to_csv(OUT_FULL,  index=False)
arch.to_csv(OUT_ARCH,  index=False)

print(f'\u2705 Full detail   → {OUT_FULL}')
print(f'\u2705 By arch       → {OUT_ARCH}')
print(f'\nAll saved to: {ANALYSIS_DIR}')
